# Rolling Summarization [Step 02.02]

> **MLCourse - Agentic AI - Agent Patterns**

The standard answer to an overgrown conversation:

```
   +----------------------------------+----------------------+
   |          OLD TURNS               |    RECENT TURNS      |
   |     (compressed to a summary)    |   (kept verbatim)    |
   +----------------------------------+----------------------+
                    ^                            ^
                    |                            |
             bounded in size              bounded in count
```

Keep the last *k* turns exactly as they were, and replace everything older with
a running summary that is **rewritten** each time the buffer overflows.

### What you'll learn

- The rolling-summary loop, implemented from scratch in about fifteen lines.
- Why the summary must be **rewritten**, not appended to.
- The failure mode nobody warns you about: **compounding drift**.
- Measured recall vs. truncation and vs. the full transcript.

### Why it matters

This is what `ConversationSummaryBufferMemory` and every framework equivalent do
under the hood. Building it yourself takes ten minutes and tells you exactly
which knobs matter - buffer size, summary budget, and the summarisation prompt,
which is by far the most important of the three and the one frameworks hide.

### Prerequisites

- [01_when_the_window_overflows](01_when_the_window_overflows.ipynb)

### Setup: environment, model, token counting, rate-limit-aware call helper


In [ ]:
import os                              # environment variables
import time                            # timing and pacing
import json                            # pretty-printing structured context
from pathlib import Path               # locating the track root
from dotenv import load_dotenv         # reads KEY=value pairs from .env

# Walk UP from the notebook folder until we hit the repo root, then load the
# (gitignored) .env that lives inside 03_agentic_ai. Note the extra path
# segment: the walk-up lands on the REPO ROOT, not on the track folder.
TRACK = Path.cwd()
while not (TRACK / "03_agentic_ai").exists() and TRACK != TRACK.parent:
    TRACK = TRACK.parent
load_dotenv(TRACK / "03_agentic_ai" / ".env")

GROQ_MODEL = "qwen/qwen3.8-27b"        # hosted, fast, generous free tier
# Local alternative (documented, not used here): Ollama `llama3.1:8b` via
# `from langchain_ollama import ChatOllama`. OpenAI is never used in this course.

from langchain_groq import ChatGroq


def make_llm(temperature: float = 0.0, max_tokens: int = 300, **kw):
    """One place that constructs the chat model, so every notebook is identical."""
    return ChatGroq(model=GROQ_MODEL, temperature=temperature,
                    max_tokens=max_tokens, **kw)


# --- Token counting -----------------------------------------------------------
# Two different numbers, and it matters which one you are looking at:
#   * approx_tokens(): a LOCAL estimate using tiktoken's cl100k_base. It is not
#     the model's own tokenizer, so treat it as "within ~10%", good for
#     budgeting BEFORE you send a request.
#   * usage_metadata on the response: the provider's EXACT count. Ground truth,
#     but only available AFTER you have already paid for the call.
import tiktoken

_ENC = tiktoken.get_encoding("cl100k_base")


def approx_tokens(text) -> int:
    """Approximate token count for a string (or anything str()-able)."""
    return len(_ENC.encode(str(text)))


# --- Rate-limit-aware calling --------------------------------------------------
# The Groq free tier allows 8000 tokens per minute. Several notebooks here make
# many small calls in a loop, so we self-pace well under the ceiling and retry
# with exponential backoff if we are throttled anyway.

TPM_BUDGET = 3500                       # deliberately conservative
_WINDOW = []                            # [(timestamp, tokens), ...]
USAGE = {"calls": 0, "in": 0, "out": 0, "seconds": 0.0}


def _pace(cost: int):
    """Sleep just enough that our rolling 60s token usage stays under budget."""
    now = time.time()
    while True:
        recent = [(t, n) for (t, n) in _WINDOW if now - t < 60]
        _WINDOW[:] = recent
        if sum(n for _, n in recent) + cost <= TPM_BUDGET or not recent:
            return
        time.sleep(min(5.0, 60 - (now - recent[0][0]) + 0.5))
        now = time.time()


def chat(messages, llm=None, temperature=0.0, max_tokens=300, retries=5):
    """Send `messages`, return the AIMessage. Paces, retries, and meters usage.

    `messages` is a list of (role, content) tuples or LangChain message objects.
    """
    llm = llm or make_llm(temperature=temperature, max_tokens=max_tokens)
    est = approx_tokens(messages) + max_tokens
    delay = 4.0
    for attempt in range(retries):
        _pace(est)
        t0 = time.time()
        try:
            out = llm.invoke(messages)
        except Exception as exc:
            if "rate_limit" in str(exc) or "429" in str(exc):
                time.sleep(delay)
                delay = min(delay * 2, 45)
                continue
            raise
        u = out.usage_metadata or {}
        _WINDOW.append((time.time(), u.get("total_tokens", est)))
        USAGE["calls"] += 1
        USAGE["in"] += u.get("input_tokens", 0)
        USAGE["out"] += u.get("output_tokens", 0)
        USAGE["seconds"] += time.time() - t0
        return out
    raise RuntimeError("still rate limited after %d attempts" % retries)


def ask(prompt: str, system: str = None, **kw) -> str:
    """Convenience wrapper: one user turn in, plain text out."""
    msgs = ([("system", system)] if system else []) + [("user", prompt)]
    return chat(msgs, **kw).content.strip()


print("model:", GROQ_MODEL)
print("key loaded:", bool(os.getenv("GROQ_API_KEY")))
print("tokenizer:", "cl100k_base (approximation)")


In [2]:
# The shared conversation lives in conversation_data.py next to this notebook,
# because 40 turns pasted at the top of five notebooks would bury the lesson.
from conversation_data import (CONVERSATION, DURABLE_FACTS, EPHEMERAL_MARKERS,
                               PROBE_QUESTIONS, as_text)

print("turns          :", len(CONVERSATION))
print("transcript     : %d approx tokens" % approx_tokens(as_text()))
print("durable facts  :", len(DURABLE_FACTS))
for label, _ in DURABLE_FACTS:
    print("   -", label)

turns          : 42
transcript     : 766 approx tokens
durable facts  : 8
   - product name
   - EU-only hosting
   - budget 12,000 EUR
   - price 49 EUR/shop
   - stack: Postgres + Django
   - no third-party LLM
   - pilot: Radhaus Krueger, March
   - 30-day trial, no free tier


### A grader we will reuse in every notebook of this module


In [ ]:
# The question is never "does the summary read nicely". It is "can the agent
# still answer questions that depend on facts from the start of the thread".

def probe(memory_text: str, label: str, verbose=True):
    """Ask each probe question using ONLY `memory_text` as the agent's memory."""
    SYS = ("You are an assistant continuing a long conversation. The notes below "
           "are ALL you remember of it. Answer from the notes only. If the notes "
           "do not contain the answer, reply exactly: UNKNOWN. Be very brief.")
    hits = []
    for q, expected in PROBE_QUESTIONS:
        out = chat([("system", SYS),
                    ("user", "Your notes:\n%s\n\nQuestion: %s" % (memory_text, q))],
                   temperature=0.0, max_tokens=60)
        a = out.content.strip().lower()
        ok = any(e in a for e in expected)
        hits.append(ok)
        if verbose:
            print("  %s %-52s -> %s" % ("OK  " if ok else "LOST", q[:52],
                                        a.replace("\n", " ")[:60]))
    score = sum(hits) / len(hits)
    print("  %-22s %d/%d  (%.0f%%)  memory size: %d tokens"
          % (label, sum(hits), len(hits), score * 100, approx_tokens(memory_text)))
    return score, hits


### 1. The loop

Three moving parts:

1. A **buffer** of recent turns, kept verbatim.
2. A **summary** string covering everything older.
3. A **trigger**: when the buffer exceeds its budget, fold its oldest turns into
   the summary and drop them from the buffer.

The critical design decision is in step 3. There are two ways to fold, and only
one of them works:

| Approach | What happens |
|---|---|
| **Append** a new summary paragraph each time | The "summary" grows without bound. You have reinvented the transcript, badly. |
| **Rewrite** the whole summary given (old summary + new turns) | Bounded size, and the model can reconcile new information with old. **Do this.** |

In [4]:
SUMMARY_BUDGET = 220        # tokens the running summary is allowed to occupy
BUFFER_BUDGET = 260         # tokens of verbatim recent turns we keep

SUMMARISE_PROMPT = """You maintain the running memory of a long conversation.

Rewrite the memory below so it incorporates the new turns. The rewritten memory
must:
- Preserve every DECISION, CONSTRAINT, NAME, NUMBER and DATE. These are never
  optional, no matter how old they are.
- Drop small talk, tangents, and advice that was not acted on.
- Be written as short factual bullet points, not prose.
- Stay under {budget} tokens (roughly {words} words).

EXISTING MEMORY:
{summary}

NEW TURNS TO FOLD IN:
{turns}

Rewritten memory (bullets only, no preamble):"""


def fold(summary: str, turns: list) -> str:
    """Rewrite the summary so it also covers `turns`. Never appends."""
    prompt = SUMMARISE_PROMPT.format(budget=SUMMARY_BUDGET,
                                     words=int(SUMMARY_BUDGET * 0.7),
                                     summary=summary or "(nothing yet)",
                                     turns=as_text(turns))
    return chat([("user", prompt)], temperature=0.0,
                max_tokens=SUMMARY_BUDGET + 60).content.strip()

In [5]:
def rolling_memory(turns, verbose=True):
    """Stream the conversation through a bounded summary + buffer."""
    summary, buffer = "", []
    history = []                       # (turn_index, summary_tokens, buffer_tokens)

    for i, turn in enumerate(turns, start=1):
        buffer.append(turn)

        # Trigger: buffer is over budget -> fold its OLDEST HALF into the summary.
        if approx_tokens(as_text(buffer)) > BUFFER_BUDGET:
            fold_count = max(2, len(buffer) // 2)
            fold_count -= fold_count % 2          # keep user/assistant pairs intact
            older, buffer = buffer[:fold_count], buffer[fold_count:]
            summary = fold(summary, older)
            if verbose:
                print("turn %2d: folded %d turns -> summary now %d tokens"
                      % (i, len(older), approx_tokens(summary)))

        history.append((i, approx_tokens(summary), approx_tokens(as_text(buffer))))

    return summary, buffer, history


summary, buffer, history = rolling_memory(CONVERSATION)

turn 14: folded 6 turns -> summary now 87 tokens


turn 22: folded 8 turns -> summary now 169 tokens


turn 29: folded 6 turns -> summary now 217 tokens


turn 35: folded 6 turns -> summary now 259 tokens


turn 41: folded 6 turns -> summary now 259 tokens


> **Pitfall: splitting a user/assistant pair.** The `fold_count -= fold_count % 2`
> line keeps exchanges together. Folding a user question into the summary while
> leaving its answer in the buffer produces a buffer that starts mid-thought, and
> models handle that badly. If your turns include tool calls, the same rule
> applies with more force: a tool call and its result must move together or the
> API will reject the message list.

In [6]:
print("FINAL RUNNING SUMMARY (%d tokens):\n" % approx_tokens(summary))
print(summary)
print()
print("VERBATIM BUFFER (%d turns, %d tokens):\n" % (len(buffer), approx_tokens(as_text(buffer))))
print(as_text(buffer))

FINAL RUNNING SUMMARY (259 tokens):

*   **Project Name**: Spannerbox.
*   **Product**: Small SaaS for bike shops.
*   **Target Audience**: German bike shops.
*   **V1 Scope**: Book service appointments and track repair jobs. No other features.
*   **Platform**: Responsive web app only; no mobile app for v1.
*   **Hard Constraint**: Must run entirely inside the EU.
*   **Reasoning**: Customers reject US data hosting.
*   **Implication**: Rules out non-EU managed services; requires EU regions for any infrastructure.
*   **Budget**: 12,000 EUR for the first six months (all-in).
*   **Budget Constraint**: Tight budget requires managed hosting and avoidance of new hires.
*   **Launch Timing**: Autumn (quiet season) preferred to allow adoption before spring peak.
*   **Pricing Model**: Per-shop flat monthly fee (rejected per-seat).
*   **Pricing Amount**: 49 EUR per shop per month.
*   **Pricing Status**: Tentative; may be revisited later.
*   **Tech Stack**: Postgres and Django (locked in)

### 2. The size curve - this is the payoff

The reason to do any of this is that the working set stops growing. Let us plot
it against the unbounded baseline.

In [7]:
print("%5s %10s %9s %10s %14s %10s" % ("turn", "summary", "buffer", "working", "full history", "saving"))
print("-" * 64)
running = []
for (i, s_tok, b_tok) in history:
    running = CONVERSATION[:i]
    full = approx_tokens(as_text(running))
    work = s_tok + b_tok
    if i % 6 == 0 or i == len(history):
        print("%5d %10d %9d %10d %14d %9.0f%%"
              % (i, s_tok, b_tok, work, full, 100 * (1 - work / full)))

final_work = history[-1][1] + history[-1][2]
final_full = approx_tokens(as_text())
print()
print("working set at the end : %d tokens" % final_work)
print("full transcript        : %d tokens" % final_full)
print("compression ratio      : %.2fx" % (final_full / final_work))
print()
print("The working set is BOUNDED: it oscillates between the fold points instead")
print("of growing. That is the whole point - the curve goes flat, not down.")

 turn    summary    buffer    working   full history     saving
----------------------------------------------------------------
    6          0       130        130            130         0%
   12          0       232        232            232         0%
   18         87       209        296            339        13%
   24        169       170        339            441        23%
   30        217       179        396            551        28%
   36        259       186        445            657        32%
   42        259       182        441            766        42%

working set at the end : 441 tokens
full transcript        : 766 tokens
compression ratio      : 1.74x

The working set is BOUNDED: it oscillates between the fold points instead
of growing. That is the whole point - the curve goes flat, not down.


### 3. But did it remember anything?

A bounded working set that has forgotten the budget and the pilot customer is
not a win. Grade it, against the same probes as notebook 01.

In [8]:
WORKING_SET = "SUMMARY OF EARLIER CONVERSATION:\n%s\n\nMOST RECENT TURNS:\n%s" % (
    summary, as_text(buffer))

print("ROLLING SUMMARY + BUFFER as memory")
roll_score, _ = probe(WORKING_SET, "rolling summary")

ROLLING SUMMARY + BUFFER as memory


  OK   What is the product called?                          -> spannerbox


  OK   Where must the product be hosted, and why?           -> it must be hosted entirely inside the eu because customers r


  OK   What is the monthly price per shop?                  -> 49 eur


  OK   Which database and web framework were chosen?        -> postgres and django.


  OK   Who is the first pilot customer and when do they sta -> radhaus krueger in freiburg, starting in march.


  OK   Is it acceptable to send customer data to a hosted L -> unknown
  rolling summary        6/6  (100%)  memory size: 458 tokens


In [9]:
# The two baselines from notebook 01, recomputed here so this notebook stands alone.
print("\nFULL TRANSCRIPT")
full_score, _ = probe(as_text(), "full transcript", verbose=False)
print("\nLAST 8 TURNS (truncation)")
trunc_score, _ = probe(as_text(CONVERSATION[-8:]), "last 8 turns", verbose=False)


FULL TRANSCRIPT


  full transcript        6/6  (100%)  memory size: 766 tokens

LAST 8 TURNS (truncation)


  last 8 turns           1/6  (17%)  memory size: 142 tokens


In [10]:
print("%-22s %9s %9s %14s" % ("strategy", "tokens", "recall", "tokens/answer"))
print("-" * 58)
for name, text, s in (("full transcript", as_text(), full_score),
                      ("last 8 turns", as_text(CONVERSATION[-8:]), trunc_score),
                      ("rolling summary", WORKING_SET, roll_score)):
    n = approx_tokens(text)
    ans = s * len(PROBE_QUESTIONS)
    print("%-22s %9d %8.0f%% %14s"
          % (name, n, 100 * s, "%.0f" % (n / ans) if ans else "n/a"))
print()
print("MEASURED, this run, %s:" % GROQ_MODEL)
print("  rolling summary reached %.0f%% recall at %.0f%% of the full transcript's size."
      % (100 * roll_score, 100 * approx_tokens(WORKING_SET) / approx_tokens(as_text())))
print("  truncation reached %.0f%% recall at %.0f%%."
      % (100 * trunc_score,
         100 * approx_tokens(as_text(CONVERSATION[-8:])) / approx_tokens(as_text())))

strategy                  tokens    recall  tokens/answer
----------------------------------------------------------
full transcript              766      100%            128
last 8 turns                 142       17%            142
rolling summary              458      100%             76

MEASURED, this run, qwen/qwen3.8-27b:
  rolling summary reached 100% recall at 60% of the full transcript's size.
  truncation reached 17% recall at 19%.


### 4. Compounding drift - the failure nobody mentions

Here is the structural weakness. The summary at fold *n* is produced from the
summary at fold *n-1*, which was produced from fold *n-2*. Nothing ever
re-reads the original turns.

```
  turns 1-8  --> S1
  S1 + 9-16  --> S2      (S1's errors are now S2's facts)
  S2 + 17-24 --> S3      (and S3's, ...)
```

Any detail dropped or distorted at fold 1 is **unrecoverable**, and it will be
restated confidently forever after. This is telephone, played by a language
model, with no way to check the original.

Let us look for it: which durable facts made it into the final summary, and
which quietly evaporated?

In [11]:
low = summary.lower()
print("%-34s %s" % ("durable fact", "in final summary?"))
print("-" * 56)
lost = []
for label, markers in DURABLE_FACTS:
    kept = any(m in low for m in markers)
    if not kept:
        lost.append(label)
    print("%-34s %s" % (label, "yes" if kept else "*** DROPPED ***"))

print()
if lost:
    print("Dropped by compression: %s" % ", ".join(lost))
    print("These are unrecoverable - no later fold re-reads the original turns.")
else:
    print("All durable facts survived this run. That is a good result, but note")
    print("it is one run of a stochastic process: rerun and it can differ.")

durable fact                       in final summary?
--------------------------------------------------------
product name                       yes
EU-only hosting                    yes
budget 12,000 EUR                  yes
price 49 EUR/shop                  yes
stack: Postgres + Django           yes
no third-party LLM                 *** DROPPED ***
pilot: Radhaus Krueger, March      *** DROPPED ***
30-day trial, no free tier         *** DROPPED ***

Dropped by compression: no third-party LLM, pilot: Radhaus Krueger, March, 30-day trial, no free tier
These are unrecoverable - no later fold re-reads the original turns.


### Mitigations for drift, in order of how much they help

1. **Name the categories in the prompt.** Ours says "DECISION, CONSTRAINT, NAME,
   NUMBER, DATE" explicitly. A prompt that just says "summarise" loses numbers
   first, every time.
2. **Keep an append-only facts list beside the summary.** Constraints go in a
   list that is never rewritten, only added to. Cheap, and it removes the most
   damaging class of loss. This is essentially what notebook 04 builds.
3. **Re-summarise from source occasionally.** If you still hold the raw
   transcript (in a database, not in the window), periodically rebuild the
   summary from the original turns instead of from the previous summary.
4. **Hierarchical summaries** - notebook 03 - so the fold depth stays shallow.

### 5. Tuning the two knobs

`BUFFER_BUDGET` and `SUMMARY_BUDGET` trade against each other under a fixed
working-set size. The rules of thumb:

| Symptom | Knob |
|---|---|
| Agent loses the thread of the *current* topic | Buffer too small |
| Agent forgets old decisions | Summary too small, or prompt too vague |
| Working set too big | Shrink the buffer first - it is usually the larger half |
| Too many LLM calls | Fold less often (larger buffer, fold more turns at once) |

Note the last one: every fold is an extra LLM call. Rolling summarisation is not
free - you are spending calls to save tokens, and on a short conversation that is
a net loss. It pays off only past the point where the transcript would have grown
large, which is exactly why notebook 01 measured that point first.

### 6. Pitfalls

- **Appending instead of rewriting.** The classic bug. Your summary becomes a
  transcript with extra steps.
- **A vague summarisation prompt.** "Summarise the conversation" loses numbers,
  names and dates - the only things you actually needed.
- **Splitting exchanges.** Fold whole user/assistant pairs, and never separate a
  tool call from its result.
- **No grader.** Drift is silent by construction. If you are not probing for
  retained facts, you will not see it.
- **Using it on short conversations.** You pay an extra LLM call per fold. Below
  the crossover point, plain history is cheaper.

### Recap

| Idea | Takeaway |
|---|---|
| Summary + buffer | Compress the old, keep the recent verbatim |
| Rewrite, never append | The only version that stays bounded |
| Bounded, not smaller | The curve goes flat - that is the win |
| Compounding drift | Errors at fold 1 are permanent; name the must-keep categories |
| Costs calls | Saves tokens by spending requests; only worth it past the crossover |

**Next:** [03_hierarchical_summaries](03_hierarchical_summaries.ipynb) - keeping the
fold depth shallow, and being able to drill back down into detail.